In [136]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

## Data

### PPI

In [137]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [138]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

100%|██████████| 13715404/13715404 [00:02<00:00, 5210158.49it/s]


Number of pairs missing their reverse: 0
All pairs have their reverse present.


In [139]:
protein_interaction

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
0,9606.ENSP00000000233,9606.ENSP00000356607,173,ARF5,RALGPS2
1,9606.ENSP00000000233,9606.ENSP00000427567,154,ARF5,FHDC1
2,9606.ENSP00000000233,9606.ENSP00000253413,151,ARF5,ATP6V1E1
3,9606.ENSP00000000233,9606.ENSP00000493357,471,ARF5,CYTH2
4,9606.ENSP00000000233,9606.ENSP00000324127,201,ARF5,PSD3
...,...,...,...,...,...
13715399,9606.ENSP00000501317,9606.ENSP00000475489,195,RFX7,MPHOSPH9
13715400,9606.ENSP00000501317,9606.ENSP00000370447,158,RFX7,VCX
13715401,9606.ENSP00000501317,9606.ENSP00000312272,226,RFX7,YPEL2
13715402,9606.ENSP00000501317,9606.ENSP00000402092,169,RFX7,SAMD3


### DrugBank

In [140]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [141]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [142]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

Found 17430 drugs in the DrugBank XML.


### Genetic results

In [243]:
### import data

### genes
hpv_positive_genes  = pd.read_csv('Results/CNV results/HPV positive CNV top genes.csv')
hpv_negative_genes = pd.read_csv('Results/CNV results/HPV negative CNV top genes.csv')

### drug candiates
hpv_positive_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Direct Drug Candidates Aggregated.csv')
hpv_positive_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Indirect Drug Candidates Aggregated.csv')

hpv_negative_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Direct Drug Candidates Aggregated.csv')
#### no direct drug candidates came from Deletions, only amplifications
hpv_negative_direct_drug_candidates['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Indirect Drug Candidates Aggregated.csv')

### somatic mtuation
hpv_positive_som_genes = pd.read_csv('Results/SOM Results/HPV positive top genes.csv')
hpv_positive_som_direct_drug_candidates = pd.read_csv('Results/SOM Results/hpv_positive_som_top_direct_drug_candidates_agg.csv')
hpv_positive_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_indirect_drug_candidates = pd.read_csv('Results/SOM Results/hpv_positive_som_top_indirect_drug_candidates_agg.csv')
hpv_positive_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

hpv_negative_som_genes = pd.read_csv('Results/SOM Results/HPV negative top genes.csv')
hpv_negative_som_direct_drug_candidates = pd.read_csv('Results/SOM Results/hpv_negative_som_top_direct_drug_candidates_agg.csv')
hpv_negative_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_indirect_drug_candidates = pd.read_csv('Results/SOM Results/hpv_negative_som_top_indirect_drug_candidates_agg.csv')
hpv_negative_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

,Gene,Count,Cohort_Frequency,Normalized_Count,Normalized_Cohort_Frequency,P_Value,Adjusted_P_Value,Significant,Empirical_P_Value,Adjusted_Empirical_P_Value,frequency_percentage,mutation_score
0,TP53,405,345,0.014275,0.012160,0.000000e+00,0.000000e+00,True,0.0,0.0,0.438998,0.006267
1,CDKN2A,105,101,0.027668,0.026614,6.503132e-132,4.632181e-128,True,0.0,0.0,0.128518,0.003556
2,FAT1,133,104,0.007515,0.005876,1.187875e-94,5.640822e-91,True,0.0,0.0,0.132336,0.000994
3,PIK3CA,69,67,0.008364,0.008121,5.925466e-53,2.110355e-49,True,0.0,0.0,0.085255,0.000713
4,LRP1B,92,70,0.004749,0.003613,1.484220e-49,4.228839e-46,True,0.0,0.0,0.089072,0.000423
...,...,...,...,...,...,...,...,...,...,...,...,...
161,TEX13A,9,9,0.003667,0.003667,2.545743e-05,1.048169e-03,True,0.0,0.0,0.011452,0.000042
162,OR8J3,8,8,0.004233,0.004233,2.624608e-05,1.065990e-03,True,0.0,0.0,0.010180,0.000043
163,GABRQ,8,8,0.004219,0.004219,2.683466e-05,1.076863e-03,True,0.0,0.0,0.010180,0.000043
164,OR2G6,8,8,0.004219,0.004219,2.683466e-05,1.076863e-03,True,0.0,0.0,0.010180,0.000043


#### Literature results

In [200]:
extracted_target_df= pd.read_csv('Validation pipeline/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined = pd.read_csv('Validation pipeline/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')

In [201]:
### accumulate all genes available in drugbank or ppi
Drug_bank_genes = list(Drug_bank['gene'].values)
ppi_genes = list(protein_interaction['Translated_protein_1'].values)
all_ppi_drugbank = list(set(Drug_bank_genes + ppi_genes))

## HPV+

#### Genes

In [202]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0],
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0]
}).reset_index()
hpv_positive_combined_genes


,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ACE2,DELETION,0.0,0.025404
1,ACTL6A,AMPLIFICATION,0.0,0.011642
2,ADIPOQ,AMPLIFICATION,0.0,0.011642
3,AGO4,SOMATIC,0.010345,0.0
4,AHSG,AMPLIFICATION,0.0,0.011642
5,ANOS1,DELETION,0.0,0.025404
6,BMX,DELETION,0.0,0.025404
7,CCDC191,SOMATIC,0.001107,0.0
8,CLDN1,AMPLIFICATION,0.0,0.011642
9,CNKSR2,DELETION,0.0,0.025404


In [203]:
### merge genes with number of articles, pubmed id from literature data
hpv_positive_genes_with_lit = pd.merge(hpv_positive_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_positive_genes_with_lit.drop(columns =['INDEX'], inplace = True)

In [204]:
hpv_positive_genes_with_lit.to_csv('Results/HPV positive gene results.csv')

In [205]:
hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
8,CLDN1,AMPLIFICATION,0.0,0.011642,CLDN1,"15170668, 17091452",2.0
30,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499404e-54, 4.6307549428396854e-18","0.0116421448601379, 0.0",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
36,RFC4,AMPLIFICATION,0.0,0.011642,RFC4,16467079,1.0
41,SOX2,AMPLIFICATION,0.0,0.011642,SOX2,15942670,1.0
43,TLR7,DELETION,0.0,0.025404,TLR7,17201162,1.0


In [206]:
# hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['GENE'].isin(all_ppi_drugbank)]

#### Direct

In [207]:
### merge all hpv postive direct drug candidates
hpv_positive_final_direct = pd.concat([hpv_positive_direct_drug_candidates, hpv_positive_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value	drug_hypergeom_fdr	drug_empirical_p_value	drug_empirical_fdr	MUT_TYPE	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	GENE_SIGNIFICANT
hpv_positive_final_direct = hpv_positive_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x),
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [208]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00053,0.038913
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00053,0.038913
2,Cladribine,POLA1,DELETION,1,12,8.333333,inhibitor,chromatin binding,9.203648e-05,2.973096e-02,0.00011,0.032119
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00053,0.038913
4,Copper,"AHSG, KNG1","AMPLIFICATION, AMPLIFICATION",2,146,1.369863,None,cysteine-type endopeptidase inhibitor activity,8.218045e-15,2.566222e-11,0.00001,0.002285
5,Golotimod,TLR7,DELETION,1,5,20.000000,None,double-stranded RNA binding,1.605536e-05,6.539420e-03,0.00003,0.012774
6,NADH,NDUFB5,AMPLIFICATION,1,144,0.694444,None,NADH dehydrogenase (ubiquinone) activity,6.205177e-08,2.325204e-05,0.00001,0.002285
7,TG-100801,VEGFD,DELETION,1,8,12.500000,inhibitor,chemoattractant activity,4.038151e-05,1.576225e-02,0.00008,0.031226
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,5.225777e-06,8.159180e-04,0.00010,0.012009
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,7.766063e-05,8.661009e-03,0.00006,0.009527


In [209]:
extracted_target_df_combined

,GENE,PMID,INDEX,NUMBER_OF_ARTICLES
0,000-2,"11302242, 11302242","2610, 2610",1
1,10,"12608845, 12768769, 14967420, 15193028, 180565...","15288, 17016, 27005, 29481, 57658, 61105",6
2,106PRE,18186293,58737,1
3,106R,18186293,58737,1
4,106RECR,18186293,58737,1
...,...,...,...,...
6072,ZP-V3,12464647,13458,1
6073,ZP-V4,12464647,13458,1
6074,ZYGOMA,15883929,36459,1
6075,ZYGOMATIC,"12775236, 17522494","17081, 53057",2


In [253]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES

hpv_positive_final_direct['PMID'] = ''
hpv_positive_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    hpv_positive_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(list(set(literature_gene_targets)))
    hpv_positive_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

/var/folders/5p/swntgnbj3fbfxkx02kt3fq980000gn/T/ipykernel_2289/2193022652.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv_positive_final_direct['PMID'] = ''
/var/folders/5p/swntgnbj3fbfxkx02kt3fq980000gn/T/ipykernel_2289/2193022652.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv_positive_final_direct['NUMBER_OF_ARTICLES'] = 0
/var/folders/5p/swntgnbj3fbfxkx02kt3fq980000gn/T/ipykernel_2289/2193022652.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataF

In [254]:
hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,Target_Description
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00053,0.038913,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA,Direct
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00053,0.038913,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA,Direct
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00053,0.038913,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA,Direct
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006539,0.00003,0.012774,17201162,1,TLR7,Direct
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000005,0.000816,0.00010,0.012009,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA,Direct
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008661,0.00006,0.009527,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA,Direct


In [212]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_positive_final_direct = hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_positive_final_direct.to_csv('Results/HPV Positive direct results.csv')

#### Indirect

In [213]:
### merge all hpv positive indirect drug candidates
hpv_positive_final_indirect = pd.concat([hpv_positive_indirect_drug_candidates, hpv_positive_som_indirect_drug_candidates], ignore_index=True)
hpv_positive_final_indirect['ACTION'] = hpv_positive_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['SPECIFIC_FUNCTION'] = hpv_positive_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_hypergeom_fdr'] = hpv_positive_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_empirical_fdr'] = hpv_positive_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['MUT_TYPE'] = hpv_positive_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### aggregate/group by drug name
### columns: DRUG	CONNECTED_TO (risk gene)	
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank	
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene	
# GENE_TARGET	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr

hpv_positive_final_indirect = hpv_positive_final_indirect.groupby(['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()
hpv_positive_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(25)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
13,artenimol,"GAPDH,NPM1,ANXA2,CCT3, HSPA8,RPS13, RPS6, EEF1...","SOX2,PIK3CA,AHSG,DNAJB11,EIF4A2,FXR1,HRG,RPL39...",36,104,34.615385,ligand,aspartic-type endopeptidase inhibitor activity,0.000068,0.002285,"DELETION,AMPLIFICATION"
88,nadh,"NDUFA8, NDUFS5, NDUFS2, NDUFA2, NDUFB10, NDUFV...",NDUFB5,46,144,31.944444,binder,"NADH dehydrogenase (ubiquinone) activity, 4 ir...",0.000023,0.002285,AMPLIFICATION
35,conestat alfa,"C1R, C1S,PLAT, F12, F2,KLKB1, F11","MASP1,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,0.000454,0.002285,AMPLIFICATION
68,human c1-esterase inhibitor,"C1R, C1S,PLAT, F12, F2,KLKB1, F11","MASP1,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,0.000454,0.002285,AMPLIFICATION
127,somatostatin,"SSTR4, SSTR3, SSTR5, SSTR1, SSTR2,OPRD1,OPRM1","SST,KNG1,GNB4",7,7,100.000000,agonist,"neuropeptide binding, G protein-coupled recept...",0.000454,0.003986,AMPLIFICATION
34,clozapine,"GABBR1, CHRM1, GABBR2,CHRM3, CHRM4, CHRM5,HTR1...","SST,KNG1,GNB4",15,33,45.454545,"positive modulator, partial agonist","extracellular matrix protein binding, G protei...",0.005134,0.003986,AMPLIFICATION
32,ci-1040,"LCK, RPS6KB1, AKT1,CHEK1,PRKCA, MAPK11, MAPK1,...","PIK3CA,RFC4,GNB4,MRPL47",10,15,66.666667,inhibitor,"ATP binding, 14-3-3 protein binding",0.001983,0.003986,AMPLIFICATION
119,rizatriptan,"HTR1D, HTR1A, HTR7, HTR1F, HTR1B, HTR1E",GNB4,6,6,100.000000,agonist,"G protein-coupled serotonin receptor activity,...",0.002228,0.003986,AMPLIFICATION
55,foreskin fibroblast (neonatal),"IL1R1,IL6, TNF, IFNG,FLT1, FGFR2, FGF1, PDGFRB","TLR7,TLR8,VEGFD",8,11,72.727273,agonist,interleukin-1 binding,0.001250,0.004930,DELETION
145,trihexyphenidyl,"CHRM1,ADRB3,CHRM3, CHRM4, CHRM5,CHRM2","SST,ADIPOQ,KNG1,GNB4",6,6,100.000000,antagonist,G protein-coupled acetylcholine receptor activity,0.002228,0.005511,AMPLIFICATION


In [214]:
'artenimol' in hpv_positive_final_indirect['DRUG'].to_list()

True

In [215]:
hpv_positive_final_indirect[hpv_positive_final_indirect['ACTION']!= 'UNKNOWN'].sort_values(by = 'drug_empirical_fdr', ascending = True).head(25)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
13,artenimol,"GAPDH,NPM1,ANXA2,CCT3, HSPA8,RPS13, RPS6, EEF1...","SOX2,PIK3CA,AHSG,DNAJB11,EIF4A2,FXR1,HRG,RPL39...",36,104,34.615385,ligand,aspartic-type endopeptidase inhibitor activity,0.000068,0.002285,"DELETION,AMPLIFICATION"
68,human c1-esterase inhibitor,"C1R, C1S,PLAT, F12, F2,KLKB1, F11","MASP1,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,0.000454,0.002285,AMPLIFICATION
35,conestat alfa,"C1R, C1S,PLAT, F12, F2,KLKB1, F11","MASP1,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,0.000454,0.002285,AMPLIFICATION
88,nadh,"NDUFA8, NDUFS5, NDUFS2, NDUFA2, NDUFB10, NDUFV...",NDUFB5,46,144,31.944444,binder,"NADH dehydrogenase (ubiquinone) activity, 4 ir...",0.000023,0.002285,AMPLIFICATION
32,ci-1040,"LCK, RPS6KB1, AKT1,CHEK1,PRKCA, MAPK11, MAPK1,...","PIK3CA,RFC4,GNB4,MRPL47",10,15,66.666667,inhibitor,"ATP binding, 14-3-3 protein binding",0.001983,0.003986,AMPLIFICATION
127,somatostatin,"SSTR4, SSTR3, SSTR5, SSTR1, SSTR2,OPRD1,OPRM1","SST,KNG1,GNB4",7,7,100.000000,agonist,"neuropeptide binding, G protein-coupled recept...",0.000454,0.003986,AMPLIFICATION
119,rizatriptan,"HTR1D, HTR1A, HTR7, HTR1F, HTR1B, HTR1E",GNB4,6,6,100.000000,agonist,"G protein-coupled serotonin receptor activity,...",0.002228,0.003986,AMPLIFICATION
34,clozapine,"GABBR1, CHRM1, GABBR2,CHRM3, CHRM4, CHRM5,HTR1...","SST,KNG1,GNB4",15,33,45.454545,"positive modulator, partial agonist","extracellular matrix protein binding, G protei...",0.005134,0.003986,AMPLIFICATION
55,foreskin fibroblast (neonatal),"IL1R1,IL6, TNF, IFNG,FLT1, FGFR2, FGF1, PDGFRB","TLR7,TLR8,VEGFD",8,11,72.727273,agonist,interleukin-1 binding,0.001250,0.004930,DELETION
145,trihexyphenidyl,"CHRM1,ADRB3,CHRM3, CHRM4, CHRM5,CHRM2","SST,ADIPOQ,KNG1,GNB4",6,6,100.000000,antagonist,G protein-coupled acetylcholine receptor activity,0.002228,0.005511,AMPLIFICATION


In [216]:
hpv_positive_final_indirect[hpv_positive_final_indirect['DRUG']=='artenimol']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
13,artenimol,"GAPDH,NPM1,ANXA2,CCT3, HSPA8,RPS13, RPS6, EEF1...","SOX2,PIK3CA,AHSG,DNAJB11,EIF4A2,FXR1,HRG,RPL39...",36,104,34.615385,ligand,aspartic-type endopeptidase inhibitor activity,0.000068,0.002285,"DELETION,AMPLIFICATION"


In [217]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES
hpv_positive_final_indirect['PMID'] = ''
hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

### validate risk genes
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect['RISK_GENE_PMID'] = ''
hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        #print(gene)
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### ensure that only drugs with NUMBER_OF_ARTICLES > 0 for both drug targets and risk genes are saved, so that they have literature support
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

In [218]:
hpv_positive_final_indirect.to_csv('Results/HPV Positive indirect results.csv', index=False)

In [219]:
hpv_positive_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
0,1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3...,"SRC, LCK, LYN",PIK3CA,3,3,100.000000,UNKNOWN,ATP binding,3.802145e-03,0.012009,SOMATIC,"17961551, 17252232",2,"LYN, SRC","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.012009,SOMATIC,"15947106, 18204781",2,"JAK2, JAK3","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
3,abrocitinib,"JAK2, TYK2, JAK3, JAK1, JAK2, TYK2, JAK3, JAK1","PIK3CA, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","acetylcholine receptor binding, ATP binding, a...",3.433760e-02,0.038913,"AMPLIFICATION, SOMATIC","15947106, 18204781",4,"JAK2, JAK3","11836556, 16676365, 14581353, 16380997, 1...",34,PIK3CA
6,afatinib,"ERBB4, EGFR, ERBB2",PIK3CA,3,3,100.000000,inhibitor,"ATP binding, actin filament binding",3.802145e-03,0.012009,SOMATIC,"18309947, 12167415, 12673202, 18369180, 125838...",325,"ERBB2, ERBB4, EGFR","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
8,altiratinib,"MET, KDR, NTRK1, NTRK3",PIK3CA,4,5,80.000000,"inhibitor, antagonist",ATP binding,8.159180e-04,0.012009,SOMATIC,"11911252, 18349821, 15319714, 17075124, 111146...",15,"MET, NTRK3, NTRK1","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,xl228,"IGF1R, SRC, ABL1",PIK3CA,3,4,75.000000,UNKNOWN,"ATP binding, actin filament binding",1.400202e-02,0.021784,SOMATIC,"17961551, 15221937",2,"IGF1R, SRC","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
157,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.012009,SOMATIC,"18316583, 17047074, 18089792, 15833854, 153559...",8,MTOR,"11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
158,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.012009,SOMATIC,"16342249, 14499691, 17317803, 16630292, 167572...",9,"PDGFRA, KIT","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
159,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.012009,SOMATIC,"12743496, 18551016, 11742034, 15203751, 113705...",55,"FGFR3, FGFR1, RET","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA


In [220]:
hpv_positive_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
13,artenimol,"GAPDH,NPM1,ANXA2,CCT3, HSPA8,RPS13, RPS6, EEF1...","SOX2,PIK3CA,AHSG,DNAJB11,EIF4A2,FXR1,HRG,RPL39...",36,104,34.615385,ligand,aspartic-type endopeptidase inhibitor activity,6.753139e-05,0.002285,"DELETION,AMPLIFICATION","16316724, 17701052, 17132224, 16455286, 175658...",8,"RPL14, RPS6, RPS13, EEF1A1","11836556, 16676365, 14581353, 16380997, 1...",18,"SOX2, PIK3CA"
55,foreskin fibroblast (neonatal),"IL1R1,IL6, TNF, IFNG,FLT1, FGFR2, FGF1, PDGFRB","TLR7,TLR8,VEGFD",8,11,72.727273,agonist,interleukin-1 binding,1.250229e-03,0.004930,DELETION,"18197435, 12569015, 12548461, 11926139, 168886...",6,"FGFR2, TNF",17201162,1,TLR7
0,1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3...,"SRC, LCK, LYN",PIK3CA,3,3,100.000000,UNKNOWN,ATP binding,3.802145e-03,0.012009,SOMATIC,"17961551, 17252232",2,"LYN, SRC","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
6,afatinib,"ERBB4, EGFR, ERBB2",PIK3CA,3,3,100.000000,inhibitor,"ATP binding, actin filament binding",3.802145e-03,0.012009,SOMATIC,"18309947, 12167415, 12673202, 18369180, 125838...",325,"ERBB2, ERBB4, EGFR","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
9,amuvatinib,"MET, KIT, RET, PDGFRA, FLT3,FLT3, PDGFRA, KIT,...","PIK3CA,VEGFD, PIK3CA",6,6,100.000000,"modulator, modulator","ATP binding, ATP binding",2.228012e-03,0.012009,"DELETION,AMPLIFICATION, SOMATIC","12743496, 18551016, 15319714, 11742034, 149919...",154,"MET, RET, PDGFRA, KIT","11836556, 16676365, 14581353, 16380997, 1...",34,PIK3CA
23,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",9,9,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.012009,"DELETION,AMPLIFICATION, SOMATIC","18309947, 12167415, 12673202, 18369180, 125838...",725,"IGF1R, MET, EGFR, ERBB2, ALK, ERBB4","11836556, 16676365, 14581353, 16380997, 1...",34,PIK3CA
26,cabozantinib,"MET, KDR, RET",PIK3CA,3,3,100.000000,antagonist,ATP binding,3.802145e-03,0.012009,SOMATIC,"12743496, 18551016, 15319714, 11742034, 149919...",58,"MET, RET","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
28,capivasertib,"AKT3, AKT2, AKT1",PIK3CA,3,3,100.000000,inhibitor,"ATP binding, 14-3-3 protein binding",3.802145e-03,0.012009,SOMATIC,"14581353, 12532415, 15896313, 17974918, 160025...",8,"AKT1, AKT2","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
38,dacomitinib,"ERBB4, EGFR, ERBB2",PIK3CA,3,3,100.000000,"antagonist, inhibitor","ATP binding, actin filament binding",3.802145e-03,0.012009,SOMATIC,"18309947, 12167415, 12673202, 18369180, 125838...",325,"ERBB2, ERBB4, EGFR","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
41,debio-1347,"FGFR1, FGFR3, FGFR2",PIK3CA,3,3,100.000000,inhibitor,ATP binding,3.802145e-03,0.012009,SOMATIC,"17545628, 16807070, 11605053, 12569015, 151653...",10,"FGFR3, FGFR1, FGFR2","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA


#### overall

In [221]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00053,0.038913,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00053,0.038913,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00053,0.038913,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006539,0.00003,0.012774,17201162,1,TLR7
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000005,0.000816,0.00010,0.012009,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008661,0.00006,0.009527,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA


In [222]:
hpv_positive_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
0,1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3...,"SRC, LCK, LYN",PIK3CA,3,3,100.000000,UNKNOWN,ATP binding,3.802145e-03,0.012009,SOMATIC,"17961551, 17252232",2,"LYN, SRC","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.012009,SOMATIC,"15947106, 18204781",2,"JAK2, JAK3","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
3,abrocitinib,"JAK2, TYK2, JAK3, JAK1, JAK2, TYK2, JAK3, JAK1","PIK3CA, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","acetylcholine receptor binding, ATP binding, a...",3.433760e-02,0.038913,"AMPLIFICATION, SOMATIC","15947106, 18204781",4,"JAK2, JAK3","11836556, 16676365, 14581353, 16380997, 1...",34,PIK3CA
6,afatinib,"ERBB4, EGFR, ERBB2",PIK3CA,3,3,100.000000,inhibitor,"ATP binding, actin filament binding",3.802145e-03,0.012009,SOMATIC,"18309947, 12167415, 12673202, 18369180, 125838...",325,"ERBB2, ERBB4, EGFR","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
8,altiratinib,"MET, KDR, NTRK1, NTRK3",PIK3CA,4,5,80.000000,"inhibitor, antagonist",ATP binding,8.159180e-04,0.012009,SOMATIC,"11911252, 18349821, 15319714, 17075124, 111146...",15,"MET, NTRK3, NTRK1","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,xl228,"IGF1R, SRC, ABL1",PIK3CA,3,4,75.000000,UNKNOWN,"ATP binding, actin filament binding",1.400202e-02,0.021784,SOMATIC,"17961551, 15221937",2,"IGF1R, SRC","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
157,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.012009,SOMATIC,"18316583, 17047074, 18089792, 15833854, 153559...",8,MTOR,"11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
158,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.012009,SOMATIC,"16342249, 14499691, 17317803, 16630292, 167572...",9,"PDGFRA, KIT","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA
159,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.012009,SOMATIC,"12743496, 18551016, 11742034, 15203751, 113705...",55,"FGFR3, FGFR1, RET","11836556, 16676365, 14581353, 16380997, 1...",17,PIK3CA


In [223]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_positive_final_direct['Target_Description'] = 'Direct'
hpv_positive_final_indirect['Target_Description'] = 'Indirect'
hpv_positive_final_results = pd.concat([hpv_positive_final_direct, hpv_positive_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_positive_final_results = hpv_positive_final_results.fillna('NA')
hpv_positive_final_results = hpv_positive_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

/var/folders/5p/swntgnbj3fbfxkx02kt3fq980000gn/T/ipykernel_2289/4020300996.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv_positive_final_direct['Target_Description'] = 'Direct'


In [224]:
hpv_positive_final_results.sort_values(by = 'PERCENTAGE_OF_TARGETS_HIT', ascending = False).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),NUM_DIRECT_TARGETS_HIT,Number of risk or immediate neighbor genes targeted,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,Target_Description
0,1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3...,"SRC, LCK, LYN",PIK3CA,NA,3.0,NA,100.0,UNKNOWN,ATP binding,3.802145e-03,0.012009,Indirect
56,pd-168393,"SRC, EGFR, ERBB2",PIK3CA,NA,3.0,NA,100.0,inhibitor,"ATP binding, actin filament binding",3.802145e-03,0.012009,Indirect
37,gilteritinib,"AXL, ALK, FLT3,HTR1A, AXL, ALK, FLT3","PIK3CA,GNB4, PIK3CA",NA,4.0,NA,100.0,"inhibitor, inhibitor","ATP binding, ATP binding",3.433760e-02,0.038913,Indirect
38,gsk-1059615,"PIK3CG, PIK3C3, MTOR",PIK3CA,NA,3.0,NA,100.0,inhibitor,"1-phosphatidylinositol-3-kinase activity, ATP ...",3.802145e-03,0.021784,Indirect
40,infigratinib,"FGFR1, FGFR4, FGFR3, FGFR2,FGFR1,FGFR2, FGFR4,...","PIK3CA,ANOS1,VEGFD, PIK3CA",NA,4.0,NA,100.0,"inhibitor, inhibitor","ATP binding, ATP binding",3.433760e-02,0.038913,Indirect
42,jnj-28312141,"LCK, KIT, NTRK3, FLT3, LCK, KIT, NTRK3, FLT3","PIK3CA, PIK3CA",NA,4.0,NA,100.0,"inhibitor, inhibitor","ATP binding, ATP binding",3.433760e-02,0.038913,Indirect
43,lenvatinib,"FGFR1, FGFR4, FGFR3, KDR, FLT1, KIT, RET, PDGF...","PIK3CA,ANOS1,VEGFD, PIK3CA",NA,10.0,NA,100.0,"inhibitor, inhibitor","ATP binding, ATP binding",3.033567e-06,0.012009,Indirect
44,linifanib,"KDR, FLT1, KIT, CSF1R, FLT4, FLT3,FLT1, FLT3, ...","PIK3CA,VEGFD, PIK3CA",NA,6.0,NA,100.0,"modulator, modulator","ATP binding, ATP binding",2.228012e-03,0.012009,Indirect
46,m-2698,"AKT3, RPS6KB1, AKT1",PIK3CA,NA,3.0,NA,100.0,inhibitor,"ATP binding, 14-3-3 protein binding",3.802145e-03,0.012009,Indirect
47,midostaurin,"KDR, PDGFRB, KIT, PDGFRA, FLT3,PRKCA, PRKCG, K...","PIK3CA,GNB4, PIK3CA",NA,7.0,NA,100.0,"antagonist, antagonist","ATP binding, ATP binding",4.544467e-04,0.012009,Indirect


## HPV-

#### Genes

In [247]:
hpv_negative_som_genes

,Gene,Count,Cohort_Frequency,Normalized_Count,Normalized_Cohort_Frequency,P_Value,Adjusted_P_Value,Significant,Empirical_P_Value,Adjusted_Empirical_P_Value,frequency_percentage,mutation_score
0,TP53,405,345,0.014275,0.012160,0.000000e+00,0.000000e+00,True,0.0,0.0,0.438998,0.006267
1,CDKN2A,105,101,0.027668,0.026614,6.503132e-132,4.632181e-128,True,0.0,0.0,0.128518,0.003556
2,FAT1,133,104,0.007515,0.005876,1.187875e-94,5.640822e-91,True,0.0,0.0,0.132336,0.000994
3,PIK3CA,69,67,0.008364,0.008121,5.925466e-53,2.110355e-49,True,0.0,0.0,0.085255,0.000713
4,LRP1B,92,70,0.004749,0.003613,1.484220e-49,4.228839e-46,True,0.0,0.0,0.089072,0.000423
...,...,...,...,...,...,...,...,...,...,...,...,...
161,TEX13A,9,9,0.003667,0.003667,2.545743e-05,1.048169e-03,True,0.0,0.0,0.011452,0.000042
162,OR8J3,8,8,0.004233,0.004233,2.624608e-05,1.065990e-03,True,0.0,0.0,0.010180,0.000043
163,GABRQ,8,8,0.004219,0.004219,2.683466e-05,1.076863e-03,True,0.0,0.0,0.010180,0.000043
164,OR2G6,8,8,0.004219,0.004219,2.683466e-05,1.076863e-03,True,0.0,0.0,0.010180,0.000043


In [256]:
### combine hpv negative somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0],
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0]
}).reset_index()
# hpv_negative_combined_genes.sort_values(by='q_value')

In [259]:
hpv_negative_combined_genes[hpv_negative_combined_genes['gene_name']=='PIK3CA']

,gene_name,MUT_TYPE,q_value,empirical_q_value
165,PIK3CA,"AMPLIFICATION, SOMATIC","0.0, 2.110354630735758e-49","0.0038596926621909, 0.0"


In [260]:
### merge genes with number of articles, pubmed id from literature data
hpv_negative_gene_results_with_lit = pd.merge(hpv_negative_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_negative_gene_results_with_lit.drop(columns = ['INDEX'], inplace = True)

In [261]:
hpv_negative_gene_results_with_lit = hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['NUMBER_OF_ARTICLES']>0]

In [262]:
hpv_negative_gene_results_with_lit.sort_values(by = ['empirical_q_value', 'q_value'], ascending=[True, True], inplace=True)

In [263]:
hpv_negative_gene_results_with_lit.to_csv('Results/HPV negative gene results.csv')

In [264]:
hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['GENE'].isin(Drug_bank_genes)]

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
204,TP53,SOMATIC,0.0,0.0,TP53,"11390535, 11445847, 11445859, 11916556, 125898...",29.0
81,HRAS,SOMATIC,0.0,0.0,HRAS,16676365,1.0
53,EPHA2,SOMATIC,0.0,0.0,EPHA2,"12494475, 16309192, 18030354, 18425361, 18485799",5.0
177,REG1A,SOMATIC,0.0,0.0,REG1A,12901795,1.0
162,PDE3A,SOMATIC,0.0,0.0,PDE3A,15078486,1.0
32,COL1A2,SOMATIC,0.0,0.0,COL1A2,18254958,1.0
175,RAC1,SOMATIC,0.000046,0.0,RAC1,"17234718, 17592548",2.0
171,PRKCI,AMPLIFICATION,0.0,0.00386,PRKCI,17990328,1.0
165,PIK3CA,"AMPLIFICATION, SOMATIC","0.0, 2.110354630735758e-49","0.0038596926621909, 0.0",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0


#### Direct

In [231]:
### combine all hpv negative direct drug candidates
hpv_negative_final_direct = pd.concat([hpv_negative_direct_drug_candidates, hpv_negative_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value      

hpv_negative_final_direct = hpv_negative_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x.unique()), ### unique mutation types per drug
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [232]:
for gene in hpv_negative_final_direct['GENE_TARGET']:
    gene = gene.split(',')
    if 'PIK3CA' in gene:
        print(gene)

['PIK3CA']
['PIK3CA']
['PIK3CA']


In [233]:
### validate hpv negative direct drug candidates with literature data
hpv_negative_final_direct['PMID'] = ''
hpv_negative_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

    

In [234]:
hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0].sort_values(by='NUMBER_OF_ARTICLES', ascending=False)

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
1,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,6.224881e-07,0.000486,0.00010,0.029272,"12673679, 12702551, 16455286, 16713672, 159659...",29,TP53
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,2.553685e-04,0.047846,0.00020,0.043568,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
13,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00045,0.047880,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
14,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00046,0.047880,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
9,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA2","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,1.720246e-09,0.000002,0.00001,0.002755,"12494475, 18425361, 17990328, 18030354, 184857...",6,"PRKCI, EPHA2"
6,Dasatinib,EPHA2,SOMATIC,1.0,23.0,4.347826,antagonist,ATP binding,1.350788e-06,0.000744,0.00010,0.029272,"12494475, 18425361, 18030354, 18485799, 16309192",5,EPHA2
12,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,5.157973e-08,0.000104,0.00010,0.029272,"12494475, 18425361, 18030354, 18485799, 16309192",5,EPHA2
0,3-isobutyl-1-methyl-7H-xanthine,PDE3A,SOMATIC,1.0,15.0,6.666667,inhibitor,"3',5'-cGMP-inhibited cyclic-nucleotide phospho...",2.553685e-04,0.047846,0.00020,0.043568,15078486,1,PDE3A
3,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.278605e-07,0.000080,0.00001,0.002755,17990328,1,PRKCI


In [235]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_negative_final_direct= hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_negative_final_direct.to_csv('Results/HPV Negative direct results.csv')

#### Indirect

In [236]:
### combine all hpv negative indirect drug candidates
hpv_negative_final_indirect = pd.concat([hpv_negative_indirect_drug_candidates, hpv_negative_som_indirect_drug_candidates])
hpv_negative_final_indirect['ACTION'] = hpv_negative_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['SPECIFIC_FUNCTION'] = hpv_negative_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_hypergeom_fdr'] = hpv_negative_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_empirical_fdr'] = hpv_negative_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['MUT_TYPE'] = hpv_negative_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### group by drug and comma seperate genes and mutation type
### columns: DRUG	CONNECTED_TO (risk gene)
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank  
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene  
# GENE_TARGET	
# GENE_Cohort_Frequency   
# GENE_Normalized_Count	
# GENE_Normalized_Cohort_Frequency

hpv_negative_final_indirect = hpv_negative_final_indirect.groupby (['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()

In [237]:
'artenimol' in hpv_negative_final_indirect['DRUG']

False

In [238]:
hpv_negative_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(20)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
72,quercetin,"HCK, PIK3CG, ESR1, STAT3, HSP90AA1,LPCAT1, LPC...","PIK3CA,PLD1,ADIPOQ",19,32,59.375000,inhibitor,"ATP binding, 1-phosphatidylinositol-3-kinase a...",0.000915,0.002755,AMPLIFICATION
54,nadh,"NDUFA8, NDUFS5, NDUFS2, NDUFA2, NDUFB10, NDUFV...",NDUFB5,58,144,40.277778,binder,"NADH dehydrogenase (ubiquinone) activity, 4 ir...",0.000036,0.002755,AMPLIFICATION
90,tivozanib,"FGFR1, MET, KDR, PDGFRB, FLT1, KIT, PDGFRA, FL...",PIK3CA,10,11,90.909091,inhibitor,ATP binding,0.000562,0.002755,AMPLIFICATION
21,cholic acid,"PLA2G1B,CES1","PLD1,NCEH1",16,22,72.727273,inhibitor,bile acid binding,0.000136,0.002755,AMPLIFICATION
13,biotin,"ACACA, HLCS, PC,ACACB, MCCC1, PCCB, PCCA, MCCC2","MCCC1,EHHADH",8,8,100.000000,"cofactor, substrate","acetyl-CoA carboxylase activity, ATP binding",0.001160,0.002755,AMPLIFICATION
83,somatostatin,"OPRM1,SSTR4, SSTR3, SSTR5, SSTR1, SSTR2,OPRD1","GNB4,SST,KNG1",7,7,100.000000,inhibitor,beta-endorphin receptor activity,0.004426,0.004804,AMPLIFICATION
81,semaxanib,"FGFR1, KDR, PDGFRB, FLT1, KIT, RET, PDGFRA,MPL","PIK3CA,THPO",8,9,88.888889,inhibitor,ATP binding,0.006129,0.004804,AMPLIFICATION
71,purvalanol,"CDK1, CDK5, MAPK1,CDK2, CDK4","CDKN2A,CDKN2B",5,9,55.555556,"binder, inhibitor","ATP binding, acetylcholine receptor activator ...",0.001660,0.005511,DELETION
58,olomoucine,"CDK1, CDK5, MAPK1,CDK2","CDKN2A,CDKN2B",4,4,100.000000,"binder, inhibitor","ATP binding, acetylcholine receptor activator ...",0.000603,0.005511,DELETION
7,alvocidib,"CDK2, CDK4, CDK6",CDKN2B,6,12,50.000000,inhibitor,ATP binding,0.000544,0.005511,DELETION


In [239]:
### add in literature validation columns
hpv_negative_final_indirect['PMID'] = ''
hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles


### validate risk genes
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect['RISK_GENE_PMID'] = ''
hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### make sure only drugs with literature support for both drug targets and risk genes are saved
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

hpv_negative_final_indirect.to_csv('Results/HPV Negative indirect results.csv', index=False)



In [240]:
hpv_negative_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)[['DRUG', 'drug_empirical_fdr','LITERATURE_GENE_TARGETS', 'RISK_GENE_PMID','RISK_GENE_LITERATURE_GENE_TARGETS','RISK_GENE_NUMBER_OF_ARTICLES','MUT_TYPE']]

,DRUG,drug_empirical_fdr,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_LITERATURE_GENE_TARGETS,RISK_GENE_NUMBER_OF_ARTICLES,MUT_TYPE
90,tivozanib,0.002755,"MET, FGFR1, PDGFRA, KIT","15586224, 16815198, 14581353, 17549376, 179903...",PIK3CA,17,AMPLIFICATION
72,quercetin,0.002755,"HCK, SHBG, ESR1, STAT3","15586224, 16815198, 14581353, 17549376, 179903...",PIK3CA,17,AMPLIFICATION
81,semaxanib,0.004804,"RET, FGFR1, PDGFRA, KIT","15586224, 16815198, 14581353, 17549376, 179903...",PIK3CA,17,AMPLIFICATION
33,fn-1501,0.005511,"CDK2, CDK4, CDK6","11309301, 12684406, 14654786, 15495191, 149917...","CDKN2A, CDKN2B",18,DELETION
58,olomoucine,0.005511,"CDK2, CDK5, CDK1","11309301, 12684406, 14654786, 15495191, 149917...","CDKN2A, CDKN2B",18,DELETION
71,purvalanol,0.005511,"CDK2, CDK5, CDK4, CDK1","11309301, 12684406, 14654786, 15495191, 149917...","CDKN2A, CDKN2B",18,DELETION
7,alvocidib,0.005511,"CDK2, CDK4, CDK6","17673925, 12632081",CDKN2B,2,DELETION
49,martinostat,0.006536,HDAC1,"11224600, 14685876, 11420458, 17429099",BCL6,4,AMPLIFICATION
43,hymenialdisine,0.014052,"CDK2, CDK5, CDK1","11309301, 12684406, 14654786, 15495191, 149917...","CDKN2A, CDKN2B",18,DELETION
85,su9516,0.014052,"CDK2, CDK5, CDK1","11309301, 12684406, 14654786, 15495191, 149917...","CDKN2A, CDKN2B",18,DELETION


#### overall

In [241]:
hpv_negative_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,3-isobutyl-1-methyl-7H-xanthine,PDE3A,SOMATIC,1.0,15.0,6.666667,inhibitor,"3',5'-cGMP-inhibited cyclic-nucleotide phospho...",2.553685e-04,0.047846,0.00020,0.043568,15078486,1,PDE3A
1,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,6.224881e-07,0.000486,0.00010,0.029272,"12673679, 12702551, 16455286, 16713672, 159659...",29,TP53
3,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.278605e-07,0.000080,0.00001,0.002755,17990328,1,PRKCI
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,2.553685e-04,0.047846,0.00020,0.043568,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
6,Dasatinib,EPHA2,SOMATIC,1.0,23.0,4.347826,antagonist,ATP binding,1.350788e-06,0.000744,0.00010,0.029272,"12494475, 18425361, 18030354, 18485799, 16309192",5,EPHA2
9,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA2","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,1.720246e-09,0.000002,0.00001,0.002755,"12494475, 18425361, 17990328, 18030354, 184857...",6,"PRKCI, EPHA2"
12,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,5.157973e-08,0.000104,0.00010,0.029272,"12494475, 18425361, 18030354, 18485799, 16309192",5,EPHA2
13,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00045,0.047880,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
14,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00046,0.047880,"15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA


In [242]:
hpv_negative_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
2,acetylsalicylic acid,"PRKAA1, PCNA, CASP3, CASP1, IKBKB, NFKBIA, PTG...","TP53, HRAS, PIK3CA, HAS2, UGT2B4",15,19,78.947368,"activator, downregulator, inhibitor, inhibitor...",[hydroxymethylglutaryl-CoA reductase (NADPH)] ...,0.000486,0.029272,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC","12702551, 16954163, 11263155, 11834097, 146666...",114,"PTGS2, TNFAIP6, NFKBIA, AKR1C1, TP53, MYC, CCN...","16815198, 17549376, 12673679, 17989125, 127025...",47,"TP53, HRAS, PIK3CA"
4,ag-24322,"CDK1,CDK2, CDK4","CDKN2A,CDKN2B",3,3,100.000000,inhibitor,ATP binding,0.010932,0.023420,DELETION,"15645429, 11180160, 12017338, 12747978, 155432...",29,"CDK2, CDK4, CDK1","11309301, 12684406, 14654786, 15495191, 149917...",18,"CDKN2A, CDKN2B"
5,alsterpaullone,"CDK1, CDK5,CDK2","CDKN2A,CDKN2B",3,4,75.000000,inhibitor,"ATP binding, acetylcholine receptor activator ...",0.034197,0.036802,DELETION,"12480548, 15645429, 11272092, 11585773, 120173...",16,"CDK2, CDK5, CDK1","11309301, 12684406, 14654786, 15495191, 149917...",18,"CDKN2A, CDKN2B"
6,altiratinib,"MET, KDR, NTRK1, NTRK3,TEK","PIK3CA,THPO",5,5,100.000000,"inhibitor, antagonist",ATP binding,0.046272,0.047880,AMPLIFICATION,"11911252, 18349821, 15319714, 17075124, 111146...",15,"MET, NTRK3, NTRK1","15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
7,alvocidib,"CDK2, CDK4, CDK6",CDKN2B,6,12,50.000000,inhibitor,ATP binding,0.000544,0.005511,DELETION,"11180160, 12017338, 12747978, 15543230, 124476...",33,"CDK2, CDK4, CDK6","17673925, 12632081",2,CDKN2B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,tetrabromo-2-benzotriazole,"MAP2K1, LCK, RPS6KB1, AKT1,PRKCA, MAPK11, MAPK...","PIK3CA,GNB4,DVL3,RFC4,MECOM,CDKN2A, TP53, PIK3...",13,18,72.222222,"inhibitor, inhibitor, inhibitor, inhibitor","ATP binding, 14-3-3 protein binding, [hydroxym...",0.001380,0.029272,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC","14581353, 12532415, 15896313, 17974918, 160025...",24,"MAPK8, AKT1, CHEK1","11309301, 16815198, 17549376, 14654786, 126736...",82,"CDKN2A, DVL3, RFC4, TP53, PIK3CA"
90,tivozanib,"FGFR1, MET, KDR, PDGFRB, FLT1, KIT, PDGFRA, FL...",PIK3CA,10,11,90.909091,inhibitor,ATP binding,0.000562,0.002755,AMPLIFICATION,"15319714, 14991901, 11705871, 17253141, 117524...",25,"MET, FGFR1, PDGFRA, KIT","15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
93,trilaciclib,"CDK5,CDK2, CDK4, CDK6, CDK7, CDK6, CDK9, CDK5,...","CDKN2A,CDKN2B, TP53, PIK3CA",4,6,66.666667,"inhibitor, inhibitor, inhibitor","acetylcholine receptor activator activity, ATP...",0.028340,0.029272,"DELETION, SOMATIC, SOMATIC","11180160, 12017338, 12747978, 15543230, 124476...",68,"CDK2, CDK5, CDK4, CDK6","11309301, 16815198, 14654786, 17549376, 126736...",64,"TP53, CDKN2A, CDKN2B, PIK3CA"
97,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,0.028340,0.029272,SOMATIC,"12743496, 18551016, 11742034, 15203751, 113705...",55,"FGFR3, FGFR1, RET","15586224, 16815198, 14581353, 17549376, 179903...",17,PIK3CA
